# Chapter 4. Quantum mechanics and basis functions

Quantum chemistry models electrons explicitly, allowing us to study bonding and electronic energies. This chapter connects the equations to a small, reproducible numerical experiment: representing a hydrogen-like 1s function with contracted Gaussians.

**Learning objectives**

- Interpret a wavefunction, probability density, and expectation value.
- Identify every term and unit convention in the molecular Hamiltonian.
- Explain what the Born-Oppenheimer approximation neglects.
- Distinguish a many-electron wavefunction, a molecular orbital, and a basis function.
- Normalize primitive and contracted basis functions in three dimensions.
- Compare STO-3G and STO-6G using radial probabilities and explicit numerical criteria.

**Prerequisites:** Chapters 1-3, basic differentiation/integration, and matrix multiplication. Run this notebook from top to bottom in the course environment. All basis data are embedded locally; execution needs NumPy, SciPy, and Matplotlib, with no quantum-chemistry engine or network connection.

### Start with a question

Why does a bond have a preferred length, and why can a molecule absorb only particular photon energies? A classical ball-and-spring picture is useful, but the electronic states that create bonds require a quantum description. This chapter builds the vocabulary needed to ask a computer for those states without confusing a picture with a probability or a basis function with an electron.

**A first reading:** follow the state/probability picture, the five energy terms, the separation of electronic and nuclear motion, and the basis plots. The normalization/parser details and matrix integrals are marked **deeper reading**: run their cells to produce the results, but you need not derive every line before understanding the conclusions.

**Vocabulary to keep nearby:** a *state* is a complete quantum description within the chosen model; an *amplitude* is a number that can interfere with another amplitude; a *probability density* is its squared magnitude per coordinate volume; an *operator* is a rule that acts on a function; a *basis* is a set of building-block functions used to represent a state or an orbital.


In [ ]:
import os
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"
from pathlib import Path
import sys
import numpy as np
import scipy
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from scipy import constants
from scipy.integrate import quad
from scipy.optimize import minimize_scalar
from scipy.linalg import eigh

OUTPUT_DIR = Path("outputs/chapter04")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__, "SciPy:", scipy.__version__)


## 4.1. The Schrodinger equation and quantum states

The Schrodinger equation tells us how a quantum state evolves. For a nonrelativistic system, the time-dependent Schrodinger equation is

$$i\hbar\frac{\partial\Psi(q,t)}{\partial t}=\hat H\Psi(q,t).$$

Here $i=\sqrt{-1}$, $t$ is time, $\hbar$ is Planck's constant divided by $2\pi$, and $\hat H$ is the Hamiltonian (the total-energy operator). The symbol $q$ denotes all coordinates required to describe the system, including spin where relevant. For a time-independent Hamiltonian, a stationary state has

$$\hat H\psi_n(q)=E_n\psi_n(q),\qquad
\Psi_n(q,t)=\psi_n(q)e^{-iE_nt/\hbar}.$$

The index $n$ labels an allowed stationary state; $E_n$ is its energy. An *eigenstate* is a function whose shape is unchanged by an operator, apart from multiplication by its eigenvalue (here, the energy).

The time-dependent phase does not change $|\Psi_n|^2$. A superposition of different energy eigenstates can have a time-dependent probability density. Solving the stationary equation is therefore a particular task, not a statement that every quantum state is stationary.

The Hamiltonian, boundary conditions, particle symmetry, and normalization together define the problem. [MIT physical chemistry lecture materials](https://ocw.mit.edu/courses/5-61-physical-chemistry-fall-2007/pages/lecture-notes/).

### Wavefunction and probability

For one particle in position space, $\psi(\mathbf r)$ is a probability **amplitude**, which may be complex. For a normalized bound state,

$$\int_{\mathbb R^3}|\psi(\mathbf r)|^2\,d^3r=1.$$

The probability of finding the particle in a region $\Omega$ is the integral over that region. The density $|\psi|^2$ is not itself a probability at a single point: it has units of inverse volume. Momentum probabilities use the wavefunction's momentum-space representation, obtained by a Fourier transform.

For $N$ electrons, the electronic wavefunction depends on all $N$ space-spin coordinates: $\Psi(\mathbf x_1,\ldots,\mathbf x_N)$. It is not generally a single function of one three-dimensional position. Integrating out the other coordinates, summing over spin, and multiplying by $N$ gives a one-electron number density whose integral is $N$.

Multiplying the whole wavefunction by a constant phase leaves probabilities unchanged. Relative phases between contributions can change interference.

### See the distinction before using it: a one-dimensional box

A particle confined to $0<x<L$ by impenetrable walls is a simple **model**, not a molecular calculation. Its stationary amplitudes are $\psi_n(x)=\sqrt{2/L}\sin(n\pi x/L)$ and its energies are $E_n=n^2\pi^2\hbar^2/(2mL^2)$, for integers $n\geq1$. Here $L$ is the box length and $m$ the mass. The boundary conditions require the amplitude to vanish at both walls.

We plot the dimensionless position $u=x/L$ and amplitude $\sqrt{L}\psi_n$. The probability density in $u$ is then $L|\psi_n|^2$, with integral one. Negative amplitudes are allowed; negative probabilities are not. An internal zero is a **node**.

The third panel combines two states: $(\psi_1+e^{-i\theta}\psi_2)/\sqrt2$, where $\theta=(E_2-E_1)t/\hbar$ is their relative phase. The stationary-state probabilities stay fixed, but interference makes the superposition's probability move between regions. This illustrates why relative phase matters in spectroscopy and quantum dynamics; it predicts no particular molecule's spectrum. [MIT particle-in-a-box notes](https://www.ocw.mit.edu/courses/5-61-physical-chemistry-fall-2007/187f992fd0cde12595f74686872a4dd5_lecture8.pdf).

In [ ]:
u = np.linspace(0, 1, 1001)
box_1 = np.sqrt(2)*np.sin(np.pi*u)
box_2 = np.sqrt(2)*np.sin(2*np.pi*u)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), layout="constrained")
for state, amplitude in [(1, box_1), (2, box_2)]:
    axes[0].plot(u, amplitude, label=f"n={state}")
    axes[1].plot(u, amplitude**2, label=f"n={state}")
    np.testing.assert_allclose(np.trapezoid(amplitude**2, u), 1, atol=1e-10)
for phase, label in [(0, "relative phase 0"), (np.pi/2, "relative phase pi/2"), (np.pi, "relative phase pi")]:
    superposition = (box_1+np.exp(-1j*phase)*box_2)/np.sqrt(2)
    probability = abs(superposition)**2
    np.testing.assert_allclose(np.trapezoid(probability, u), 1, atol=1e-10)
    axes[2].plot(u, probability, label=label)
for ax, title, ylabel in zip(axes, ["Amplitudes can change sign", "Stationary probabilities", "A coherent superposition"],
                            ["Dimensionless amplitude", "Probability density in u", "Probability density in u"]):
    ax.set(xlabel="Position u = x/L", ylabel=ylabel, title=title)
    ax.axhline(0, color="0.6", linewidth=0.7)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
fig.savefig(OUTPUT_DIR / "states_and_probability.png", dpi=130, bbox_inches="tight")
plt.show()

**Pause and explain:** the second state has a node at the center, but either sign of its amplitude gives the same stationary probability. The superposition is different because it changes the relative contribution of two amplitudes before squaring. If you can explain those two statements, you have the main idea needed for orbital interference later in this chapter.

## 4.2. Operators, observables, and units

Think of an operator as an instruction: multiply the function by a position, or differentiate it to measure how it varies in space. In the position representation, common operators act as follows:

| Quantity or operation | Action on a one-particle wavefunction |
|---|---|
| Position along x | $\hat x\psi=x\psi$ |
| Gradient | $\nabla\psi=(\partial_x\psi,\partial_y\psi,\partial_z\psi)$ |
| Momentum | $\hat{\mathbf p}\psi=-i\hbar\nabla\psi$ |
| Laplacian | $\nabla^2\psi=\partial_x^2\psi+\partial_y^2\psi+\partial_z^2\psi$ |
| Kinetic energy | $\hat T\psi=-\hbar^2\nabla^2\psi/(2m)$ |
| Potential energy | $\hat V\psi=V(\mathbf r)\psi$ |

The gradient and Laplacian are mathematical differential operators; the Laplacian alone is not the kinetic energy. For a normalized state and a suitable Hermitian observable operator,

$$\langle A\rangle=\int\psi^*\hat A\psi\,d\tau.$$

If the trial function is not normalized, divide this expression by $\int|\psi|^2d\tau$. The integration includes all relevant coordinates and, when needed, sums over spin.

In the table, $m$ is the particle mass and $V$ is the potential-energy function. In the expectation value, the star denotes complex conjugation, $d\tau$ is the coordinate-volume element, and angle brackets mean a statistical mean over repeated measurements on the same state. *Hermitian* identifies the operator symmetry needed for real observable values; it does not mean that one measurement must return the mean.


### Atomic units

Use one unit system consistently. In **Hartree atomic units**, $\hbar=m_e=e=4\pi\epsilon_0=1$ in numerical expressions. Lengths are in bohr ($a_0$), energies in hartree ($E_h$), and nuclear masses in electron-mass units.

In SI, every Coulomb term contains $e^2/(4\pi\epsilon_0)$, not just $e^2$. The atomic-unit Hamiltonian below omits these factors because its units have been defined, not because the factors vanish. [NIST units and constants](https://www.nist.gov/publications/units-and-constants).

The next cell prints approximate conversions using the constants bundled with SciPy. Extra displayed digits are not a claim that a calculated molecular energy has that accuracy.

In [ ]:
bohr_angstrom = constants.physical_constants["Bohr radius"][0] / 1e-10
hartree_ev = constants.physical_constants["Hartree energy in eV"][0]
print(f"1 bohr = {bohr_angstrom:.8f} angstrom")
print(f"1 hartree = {hartree_ev:.8f} eV")

## 4.3. The molecular Hamiltonian

**Read the Hamiltonian as an energy inventory.** Moving electrons and nuclei contribute kinetic energy; like charges repel and unlike charges attract. Keep these five physical terms in mind before reading the sums.

We consider isolated, nonrelativistic point nuclei and electrons interacting by Coulomb forces, without external fields. This model omits relativistic and spin-dependent terms. Let:

- $i,j=1,\ldots,N_e$ label electrons at positions $\mathbf r_i$;
- $A,B=1,\ldots,N_n$ label nuclei at positions $\mathbf R_A$;
- $Z_A$ be nuclear charge number and $M_A$ nuclear mass in units of $m_e$;
- $r_{ij}=|\mathbf r_i-\mathbf r_j|$, $r_{iA}=|\mathbf r_i-\mathbf R_A|$, and $R_{AB}=|\mathbf R_A-\mathbf R_B|$.

In atomic units,

$$\begin{aligned}
\hat H={}&-\frac12\sum_i\nabla_i^2
-\sum_A\frac{1}{2M_A}\nabla_A^2\\
&+\sum_{i<j}\frac{1}{r_{ij}}
-\sum_{i,A}\frac{Z_A}{r_{iA}}
+\sum_{A<B}\frac{Z_AZ_B}{R_{AB}}.
\end{aligned}$$

These are electronic kinetic energy, nuclear kinetic energy, electron-electron repulsion, electron-nucleus attraction, and nucleus-nucleus repulsion. Repulsions have positive signs; attractions have negative signs.

The pair sums $i<j$ and $A<B$ count each distinct pair once and exclude self-interactions. The mixed sum visits every electron-nucleus pair. [MIT molecular Hamiltonian and MO introduction](https://www.ocw.mit.edu/courses/5-61-physical-chemistry-fall-2017/ef9da1132ff6233d732a1d6115099467_MIT5_61F17_lec24.pdf).

In [ ]:
# Bookkeeping for H2: two electrons and two nuclei.
number_electrons = 2
number_nuclei = 2
print("Electron-electron pairs:", number_electrons * (number_electrons - 1) // 2)
print("Electron-nucleus pairs:", number_electrons * number_nuclei)
print("Nucleus-nucleus pairs:", number_nuclei * (number_nuclei - 1) // 2)

# Only the nuclear repulsion is evaluated here, not the full molecular energy.
separation_bohr = 1.4
nuclear_repulsion_hartree = 1.0 / separation_bohr
print(f"H2 nuclear repulsion at {separation_bohr} bohr: {nuclear_repulsion_hartree:.6f} hartree")

## 4.4. The Born-Oppenheimer approximation

**Intuition:** choose a molecular geometry, solve its electronic problem, then ask how the energy changes when the nuclei move. Repeating this calculation builds the potential-energy surface used in Chapter 3.

Electronic structure is first solved at fixed nuclear positions $\mathbf R$. We define the electronic Hamiltonian here to **exclude** nuclear repulsion:

$$\hat H_e(\mathbf R)=-\frac12\sum_i\nabla_i^2
+\sum_{i<j}\frac1{r_{ij}}-\sum_{i,A}\frac{Z_A}{r_{iA}},$$

$$\hat H_e(\mathbf R)\Psi_k(\mathbf x;\mathbf R)
=E_{e,k}(\mathbf R)\Psi_k(\mathbf x;\mathbf R).$$

The energy surface for nuclear motion is

$$U_k(\mathbf R)=E_{e,k}(\mathbf R)+V_{NN}(\mathbf R).$$

Some programs include $V_{NN}$ in their electronic Hamiltonian or reported energy; check the convention before adding it. At one fixed geometry, $V_{NN}$ is constant with respect to electronic coordinates, but it varies when the nuclei move.

A single-surface Born-Oppenheimer model approximates the full state as an electronic factor times a nuclear factor:

$$\Phi(\mathbf x,\mathbf R)\approx\Psi_k(\mathbf x;\mathbf R)\,\chi(\mathbf R),
\qquad [\hat T_N+U_k(\mathbf R)]\chi\approx E\chi.$$

It neglects derivative couplings arising from the nuclear-coordinate dependence of the electronic state. The nuclei's larger masses motivate this separation. **Nuclear kinetic energy does not physically become zero**: it is omitted from the fixed-nuclei electronic problem and retained when treating vibrations, rotations, or nuclear dynamics.

The single-surface approximation can fail when electronic states approach one another closely, such as near conical intersections. Treating nuclei classically is an additional approximation, not the definition of Born-Oppenheimer separation. [Born and Oppenheimer, original paper](https://doi.org/10.1002/andp.19273892002).

## 4.5. Molecular orbitals and many-electron states

**Three levels of description:** a basis function is a chosen building block; an orbital is a one-electron function assembled from those blocks; a many-electron state combines electron coordinates while respecting their exchange symmetry. These are different objects, even when software displays them all as colored surfaces.

A molecular orbital (MO) $\phi_p(\mathbf r)$ is a **one-electron spatial function**. A spin orbital also includes a spin function. A many-electron wavefunction $\Psi$ is built using many space-spin coordinates and must change sign when two electrons' coordinates are exchanged.

Hartree-Fock approximates $\Psi$ by one Slater determinant of spin orbitals. Its canonical orbitals satisfy effective one-electron equations,

$$\hat f[\{\phi\}]\phi_p=\epsilon_p\phi_p,$$

where the Fock operator depends on the occupied orbitals. These equations require self-consistency. They are not the full many-electron Schrodinger equation written once per electron. Kohn-Sham DFT uses a different effective one-electron construction based on the density.

An orbital energy $\epsilon_p$ is not the molecule's total energy. Summing occupied orbital energies double counts some contributions in Hartree-Fock, among other distinctions. [Psi4 Hartree-Fock theory](https://psi4.github.io/psi4docs/master/scf.html).

### Linear combinations of basis functions

In an atom-centered basis, an MO is expanded as

$$\phi_p(\mathbf r)=\sum_{\mu=1}^{K}C_{\mu p}\chi_\mu(\mathbf r).$$

The $\chi_\mu$ are chosen basis functions, often loosely called atomic orbitals. They need not be exact isolated-atom eigenfunctions. Expanding the one-electron functions converts an operator equation into a matrix problem, such as $\mathbf F\mathbf C=\mathbf S\mathbf C\boldsymbol\epsilon$ in Hartree-Fock.

The overlap matrix is $S_{\mu\nu}=\langle\chi_\mu|\chi_\nu\rangle$. For a normalized orbital, $\mathbf c_p^\dagger\mathbf S\mathbf c_p=1$. Atom-centered functions on different centers can overlap, so squared MO coefficients are not generally independent probabilities.

Here $K$ is the number of basis functions, $C_{\mu p}$ is the coefficient of function $\mu$ in orbital $p$, $\mathbf F$ is the Fock matrix, and $\boldsymbol\epsilon$ contains orbital energies. The dagger means complex-conjugate transpose. The overlap matrix supplies the correct length/normalization measure because basis functions need not be perpendicular to one another.


### Bonding, antibonding, and nonbonding combinations

For two equivalent, normalized 1s functions $\chi_A$ and $\chi_B$ with overlap $S$, a simple model gives

$$\phi_+=\frac{\chi_A+\chi_B}{\sqrt{2(1+S)}},\qquad
\phi_-=\frac{\chi_A-\chi_B}{\sqrt{2(1-S)}}.$$

The bonding combination increases density between the nuclei; the antibonding combination has an internuclear node. A nonbonding orbital has little net effect on a particular bond. These labels concern a specified interaction and should be checked using spatial shape and energy, especially for more complicated orbitals.

In this simple case, relative signs reveal constructive or destructive interference. A global sign change of an orbital has no physical effect. Orbital occupations and total energy, including nuclear repulsion, determine whether the molecular system is bound; an attractive-looking orbital drawing alone does not establish a stable geometry. [MIT MO lecture](https://ocw.mit.edu/courses/5-61-physical-chemistry-fall-2007/resources/lecture26/).

## 4.6. Basis sets: Slater and Gaussian functions

### A normalized 1s Slater-type function

For $r=|\mathbf r|\geq0$ and exponent $\zeta>0$,

$$\chi_\zeta(r)=\left(\frac{\zeta^3}{\pi}\right)^{1/2}e^{-\zeta r}.$$

This is normalized in **three dimensions**, not by setting its maximum to one. If $r$ is measured in bohr, $\zeta$ is in bohr$^{-1}$ and $\chi$ in bohr$^{-3/2}$. For an electron bound to an infinitely massive proton in the nonrelativistic Coulomb model, the exact ground state has $\zeta=1$ bohr$^{-1}$ and energy $-0.5$ hartree.

The hydrogenic 1s function is a special case. A many-electron atom's state is not one Slater-type orbital, and general atomic orbitals can have angular structure and radial nodes.

In [ ]:
def slater_1s(radius_bohr, zeta=1.0):
    """Normalized s-type amplitude; radius in bohr, zeta in inverse bohr."""
    radius = np.asarray(radius_bohr, dtype=float)
    if not np.isfinite(zeta) or zeta <= 0:
        raise ValueError("zeta must be positive and finite.")
    if np.any(~np.isfinite(radius)) or np.any(radius < 0):
        raise ValueError("A radius must be non-negative and finite.")
    return np.sqrt(zeta**3 / np.pi) * np.exp(-zeta * radius)

print(f"Exact hydrogen 1s amplitude at the nucleus: {slater_1s(0):.6f} bohr^(-3/2)")

### Normalized primitive Gaussians

A normalized, origin-centered s-type Gaussian primitive is

$$g_\alpha(r)=N_\alpha e^{-\alpha r^2},\qquad
N_\alpha=\left(\frac{2\alpha}{\pi}\right)^{3/4}.$$

The exponent $\alpha>0$ has units bohr$^{-2}$. A large exponent produces a compact function; a small exponent produces a diffuse one. Higher angular momentum requires additional coordinate polynomials or spherical harmonics, which are outside this s-only example.

Gaussian products have convenient analytic integral formulas. This computational advantage motivates approximating orbital shapes with linear combinations of Gaussians, even though an individual Gaussian has the wrong near-nucleus and long-range behavior for a Coulombic 1s orbital. [GBasis primitive normalization](https://gbasis.qcdevs.org/_autosummary/gbasis.html).

In [ ]:
def primitive_s(radius_bohr, alpha):
    """Normalized s Gaussian with exponent alpha in bohr^(-2)."""
    radius = np.asarray(radius_bohr, dtype=float)
    if not np.isfinite(alpha) or alpha <= 0:
        raise ValueError("A Gaussian exponent must be positive and finite.")
    if np.any(~np.isfinite(radius)) or np.any(radius < 0):
        raise ValueError("A radius must be non-negative and finite.")
    return (2 * alpha / np.pi)**0.75 * np.exp(-alpha * radius**2)

### What changes when a basis set changes?

| Basis feature | What it adds |
|---|---|
| Minimal | A small set of contracted functions representing the occupied atomic shells; for H in STO-nG, one s function |
| Split valence / multiple zeta | More independently variable functions for valence orbital shapes |
| Polarization | Higher angular-momentum functions, such as p functions on H, to allow directional deformation |
| Diffuse functions | Small-exponent functions to describe more extended density |

Adding polarization or diffuse functions changes the representation; it does not by itself introduce an electron-correlation method. Accuracy depends on both the electronic-structure method and the basis.

**STO-3G and STO-6G are both minimal basis sets.** For hydrogen each supplies one contracted s function, constructed from three or six primitives respectively. Six primitives do not mean six independently optimized MOs, six occupied orbitals, or a six-zeta basis. [Psi4 basis-set overview](https://psi4.github.io/psi4docs/master/basissets.html).

**Deeper reading: how published basis parameters enter a calculation.** The central idea is that each row supplies a width and a fixed weight; the short parser handles the file notation.

### Local basis data and provenance

The following hydrogen s-shell records were checked against **Basis Set Exchange version 1** on 2026-09-15: [STO-3G data](https://www.basissetexchange.org/api/basis/sto-3g/format/json/?elements=1&version=1) and [STO-6G data](https://www.basissetexchange.org/api/basis/sto-6g/format/json/?elements=1&version=1). They are stored here so the calculation runs offline.

In the Gaussian-format excerpt, `H 0` identifies the element block. `S 3 1.00` means an s shell with three primitives and a unit scale factor. Each subsequent row gives an exponent and a contraction coefficient. `D` is Fortran scientific notation: `0.3425250914D+01` means `3.425250914`.

These coefficients multiply **normalized primitives**. They are not probabilities and should not be summed as though they were fractions. [BSE format and provenance documentation](https://molssi-bse.github.io/basis_set_exchange/bse_cli.html).

In [ ]:
sto3g_text = """
H    0
S    3   1.00
      0.3425250914D+01       0.1543289673D+00
      0.6239137298D+00       0.5353281423D+00
      0.1688554040D+00       0.4446345422D+00
"""

sto6g_text = """
H    0
S    6   1.00
      0.3552322122D+02       0.9163596281D-02
      0.6513143725D+01       0.4936149294D-01
      0.1822142904D+01       0.1685383049D+00
      0.6259552659D+00       0.3705627997D+00
      0.2430767471D+00       0.4164915298D+00
      0.1001124280D+00       0.1303340841D+00
"""

The next parser is deliberately limited to the single hydrogen s-shell excerpts above. It checks the declared number of rows and handles positive or negative `D` exponents. It rejects other shells and non-unit scale factors instead of silently misreading a general basis file. For production work, use a format-aware basis library.

In [ ]:
def parse_hydrogen_s_shell(text):
    """Read exactly one H s shell with unit scale; return alpha and d arrays."""
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if len(lines) < 3 or lines[0].split() != ["H", "0"]:
        raise ValueError("Expected an H 0 header and an s-shell block.")
    header = lines[1].split()
    if len(header) != 3 or header[0] != "S":
        raise ValueError("This teaching parser only supports one S shell.")
    number_primitives = int(header[1])
    scale_factor = float(header[2])
    if scale_factor != 1.0:
        raise ValueError("This example supports only a unit shell scale factor.")
    if number_primitives <= 0 or len(lines[2:]) != number_primitives:
        raise ValueError("The declared primitive count does not match the rows.")
    rows = []
    for line in lines[2:]:
        fields = line.split()
        if len(fields) != 2:
            raise ValueError("Each row must contain one exponent and one coefficient.")
        rows.append([float(value.replace("D", "E").replace("d", "e")) for value in fields])
    values = np.asarray(rows)
    if not np.all(np.isfinite(values)) or np.any(values[:, 0] <= 0):
        raise ValueError("All values must be finite; exponents must be positive.")
    if not np.any(values[:, 1]):
        raise ValueError("An all-zero contraction cannot be normalized.")
    return values[:, 0], values[:, 1]

basis = {}
for name, text in [("STO-3G", sto3g_text), ("STO-6G", sto6g_text)]:
    alpha, d = parse_hydrogen_s_shell(text)
    basis[name] = {"alpha": alpha, "d": d}
    print(f"{name}: {len(alpha)} primitives -> one contracted s function")
assert len(basis["STO-3G"]["alpha"]) == 3
assert len(basis["STO-6G"]["alpha"]) == 6

**Deeper reading: why overlap changes normalization.**

### Contracting and normalizing

A contracted function is a fixed weighted sum,

$$\chi(r)=N_c\sum_i d_i g_{\alpha_i}(r).$$

For normalized s primitives on the same center, the overlap is

$$S_{ij}=\int g_{\alpha_i}(\mathbf r)g_{\alpha_j}(\mathbf r)\,d^3r
=\left(\frac{2\sqrt{\alpha_i\alpha_j}}{\alpha_i+\alpha_j}\right)^{3/2}.$$

Consequently,

$$N_c=\left(\mathbf d^T\mathbf S\mathbf d\right)^{-1/2}.$$

The cross terms matter because primitives overlap. Dividing by $\sqrt{\sum_i d_i^2}$ or by the maximum curve height is incorrect. For the supplied BSE records the contraction is already normalized to rounding precision, but checking and applying the factor makes our convention explicit.

In [ ]:
def s_overlap_matrix(alpha):
    """Overlap of normalized, same-center s primitives."""
    a = alpha[:, None]
    b = alpha[None, :]
    return (2 * np.sqrt(a * b) / (a + b))**1.5


def contracted_s(radius_bohr, parameters):
    """Evaluate a contraction using its normalized weights."""
    return sum(weight * primitive_s(radius_bohr, exponent)
               for exponent, weight in zip(parameters["alpha"], parameters["weights"]))


for name, parameters in basis.items():
    overlap = s_overlap_matrix(parameters["alpha"])
    d = parameters["d"]
    norm_squared = float(d @ overlap @ d)
    if norm_squared <= 0:
        raise ValueError("Contraction has non-positive norm.")
    parameters["weights"] = d / np.sqrt(norm_squared)
    parameters["overlap"] = overlap
    print(f"{name}: raw norm squared = {norm_squared:.10f}; Nc = {norm_squared**-0.5:.10f}")
    np.testing.assert_allclose(parameters["weights"] @ overlap @ parameters["weights"], 1.0, atol=1e-12)

### A numerical normalization check

For a spherically symmetric amplitude $f(r)$,

$$\int_{\mathbb R^3}|f|^2d^3r=4\pi\int_0^\infty r^2|f(r)|^2dr.$$

We independently integrate the functions, rather than relying only on the overlap formula used to normalize them. `quad` returns an integral and an estimated numerical integration error. The latter is not the physical or basis-set error. [SciPy numerical quadrature](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.quad.html).

In [ ]:
def spherical_integral(radial_function):
    """Integrate a real radial function over 3D space, with lengths in bohr."""
    return quad(lambda radius: 4 * np.pi * radius**2 * radial_function(radius),
                0, np.inf, epsabs=1e-10, epsrel=1e-10, limit=100)


functions = {
    "Exact H 1s": lambda radius: slater_1s(radius, zeta=1.0),
    "STO zeta=1.24": lambda radius: slater_1s(radius, zeta=1.24),
}
for name, parameters in basis.items():
    # Bind each dictionary now so the functions do not all use the last basis.
    functions[name] = lambda radius, parameters=parameters: contracted_s(radius, parameters)

for name, function in functions.items():
    norm, integration_error = spherical_integral(lambda radius: abs(function(radius))**2)
    print(f"{name:14s} norm = {norm:.10f}, integration error estimate = {integration_error:.1e}")
    np.testing.assert_allclose(norm, 1.0, atol=1e-8, rtol=0)

# Also check a primitive independently of the contracted sums.
primitive_norm, _ = spherical_integral(lambda radius: primitive_s(radius, 0.7)**2)
np.testing.assert_allclose(primitive_norm, 1.0, atol=1e-8, rtol=0)

### Which Slater function are we comparing against?

STO-nG parameters were fitted and scaled for use in atoms and molecules. The supplied hydrogen contractions are not expansions of the exact isolated-H $\zeta=1$ function. For these data, a Slater function with $\zeta\approx1.24$ bohr$^{-1}$ is a closer shape reference; we verify that value by an overlap-based fit below.

Keep both references visible: **exact isolated H** and **the Slater shape approximated by these contractions**. Matching one does not imply matching the other. [BSE notes on fitting and molecular scaling](https://molssi-bse.github.io/basis_set_exchange/bse_cli.html#get-notes).

### Line cuts: how the primitive contributions add

Along the x axis, $r=|x|$. The next plot is a **line cut of an amplitude**, with a signed coordinate $x$, not a radial probability distribution. Every curve retains its three-dimensional normalization. Gray curves show weighted primitive contributions; their sum is the orange contraction.

In [ ]:
x_bohr = np.linspace(-3, 3, 1201)
radius_on_axis = np.abs(x_bohr)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True, layout="constrained")
for ax, (name, parameters) in zip(axes, basis.items()):
    for index, (alpha, weight) in enumerate(zip(parameters["alpha"], parameters["weights"])):
        ax.plot(x_bohr, weight * primitive_s(radius_on_axis, alpha),
                color="gray", alpha=0.7, linestyle=":",
                label="Weighted primitives" if index == 0 else None)
    ax.plot(x_bohr, slater_1s(radius_on_axis, 1.24), color="black", linestyle="--",
            label="Slater, zeta=1.24")
    ax.plot(x_bohr, contracted_s(radius_on_axis, parameters), color="#d55e00",
            linewidth=2, label="Normalized contraction")
    ax.set(title=f"H {name}: one contracted s function", xlabel="x (bohr)")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
axes[0].set_ylabel(r"Amplitude (bohr$^{-3/2}$)")
plt.show()

### Density at a point versus probability in a shell

For a normalized s-type amplitude, the probability density per unit volume is

$$\rho(r)=|\chi(r)|^2,$$

while the **radial probability density** is

$$P(r)=4\pi r^2|\chi(r)|^2,\qquad \Pr(r\leq |\mathbf r|<r+dr)\approx P(r)dr.$$

$\rho$ has units bohr$^{-3}$, $P$ has units bohr$^{-1}$, and $\int_0^\infty P(r)dr=1$. For exact hydrogen 1s, the spatial density is largest at the nucleus, but the radial distribution peaks at $r=1$ bohr because shells farther out have more volume. A finite density at the origin does not imply finite probability at exactly one point.

In [ ]:
radius_bohr = np.linspace(0, 5, 1201)
styles = {
    "Exact H 1s": ("#0072b2", "-"),
    "STO zeta=1.24": ("black", "--"),
    "STO-3G": ("#d55e00", "-."),
    "STO-6G": ("#009e73", ":"),
}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), layout="constrained")
for name, function in functions.items():
    density = abs(function(radius_bohr))**2
    color, linestyle = styles[name]
    axes[0].plot(radius_bohr, density, label=name, color=color, linestyle=linestyle, linewidth=2)
    axes[1].plot(radius_bohr, 4 * np.pi * radius_bohr**2 * density,
                 label=name, color=color, linestyle=linestyle, linewidth=2)
axes[0].set(title="Density per unit volume", xlabel="r (bohr)",
            ylabel=r"Density (bohr$^{-3}$)")
axes[1].set(title="Radial probability density", xlabel="r (bohr)",
            ylabel=r"P(r) (bohr$^{-1}$)")
for ax in axes:
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
plt.show()

exact_radial_probability = 4 * np.pi * radius_bohr**2 * slater_1s(radius_bohr)**2
peak_radius = radius_bohr[np.argmax(exact_radial_probability)]
mean_radius, _ = spherical_integral(lambda radius: radius * slater_1s(radius)**2)
print(f"Exact H 1s: sampled most probable radius = {peak_radius:.4f} bohr")
print(f"Exact H 1s: mean radius = {mean_radius:.4f} bohr")
assert abs(peak_radius - 1.0) < 0.005
np.testing.assert_allclose(mean_radius, 1.5, atol=1e-8, rtol=0)

### Quantifying the shape error

We compare normalized, real, positive s functions using the three-dimensional norm

$$\|f-g\|_2=\left[4\pi\int_0^\infty r^2|f(r)-g(r)|^2dr\right]^{1/2}.$$

The measure weights all spatial volume. It is different from an unweighted least-squares fit of plotted values along a line. For orbitals of arbitrary phase, first choose a consistent phase before comparing amplitudes.

We also find the Slater exponent giving the smallest error for each supplied contraction. This is a **fit to an existing basis function**, not an energy optimization or a new basis set.

In [ ]:
def squared_shape_error(first, second):
    value, _ = spherical_integral(lambda radius: abs(first(radius) - second(radius))**2)
    return value


shape_results = {}
print(f"{'Basis':8s} {'L2 to exact H':>16s} {'L2 to zeta=1.24':>18s} {'Fitted zeta':>14s}")
for name in basis:
    function = functions[name]
    error_to_h = np.sqrt(squared_shape_error(function, functions["Exact H 1s"]))
    error_to_reference = np.sqrt(squared_shape_error(function, functions["STO zeta=1.24"]))
    fit = minimize_scalar(
        lambda zeta: squared_shape_error(function, lambda radius: slater_1s(radius, zeta)),
        bounds=(0.6, 2.0), method="bounded", options={"xatol": 1e-8},
    )
    if not fit.success:
        raise RuntimeError("Slater-exponent fit did not converge.")
    shape_results[name] = {"error_to_h": error_to_h, "error_to_reference": error_to_reference,
                           "fitted_zeta": fit.x}
    print(f"{name:8s} {error_to_h:16.6f} {error_to_reference:18.6f} {fit.x:14.6f}")
    assert abs(fit.x - 1.24) < 0.01
assert shape_results["STO-6G"]["error_to_reference"] < shape_results["STO-3G"]["error_to_reference"]

**Interpret the result carefully.** STO-6G better approximates the $\zeta\approx1.24$ Slater shape under this metric. Both remain constrained to one hydrogen s basis function. The error to the exact isolated-H state need not decrease by the same amount, because the target shape differs.

A finite sum of origin-centered s Gaussians has zero radial derivative at the origin and Gaussian long-range decay. A 1s Slater function has the right-sided derivative $\chi'(0)=-\zeta\chi(0)$ and exponential decay. More Gaussians can improve the fit over a useful range without reproducing the cusp and far tail exactly. The plots and integrals expose different aspects of this limitation.

## 4.7. A variational energy check (integrals: deeper reading)

**Why care?** A function that looks plausible can still have the wrong energy. Hydrogen gives a known answer against which to check our representation and numerical machinery.

For an electron around one fixed proton, $\hat h=-\tfrac12\nabla^2-1/r$. A normalized trial function gives $E_{\rm trial}\geq-0.5$ hartree. This provides an independent physical check on the normalization and integrals.

For same-center normalized s primitives, with $p=\alpha_i+\alpha_j$,

$$T_{ij}=3\frac{\alpha_i\alpha_j}{p}S_{ij},\qquad
V_{ij}=-\frac{2\pi}{p}N_{\alpha_i}N_{\alpha_j}.$$

Thus $E=\mathbf w^T(\mathbf T+\mathbf V)\mathbf w/(\mathbf w^T\mathbf S\mathbf w)$. For the normalized Slater family, direct integration gives $E(\zeta)=\zeta^2/2-\zeta$, minimized at $\zeta=1$.

These are one-electron expectation values for fixed trial functions, not molecular Hartree-Fock or DFT calculations. [MIT variational and MO treatment](https://www.ocw.mit.edu/courses/5-61-physical-chemistry-fall-2017/ef9da1132ff6233d732a1d6115099467_MIT5_61F17_lec24.pdf).

In [ ]:
def hydrogen_matrices(parameters):
    alpha = parameters["alpha"]
    weights = parameters["weights"]
    overlap = parameters["overlap"]
    pair_exponent = alpha[:, None] + alpha[None, :]
    normalization = (2 * alpha / np.pi)**0.75
    kinetic = 3 * alpha[:, None] * alpha[None, :] / pair_exponent * overlap
    potential = -2 * np.pi * normalization[:, None] * normalization[None, :] / pair_exponent
    return kinetic + potential, overlap


def hydrogen_trial_energy(parameters):
    hamiltonian, overlap = hydrogen_matrices(parameters)
    weights = parameters['weights']
    return float(weights @ hamiltonian @ weights / (weights @ overlap @ weights))


energies = {name: hydrogen_trial_energy(parameters) for name, parameters in basis.items()}
print(f"{'Trial function':18s} {'Energy (hartree)':>18s} {'Above exact (hartree)':>24s}")
for name, energy in energies.items():
    print(f"{name:18s} {energy:18.8f} {energy + 0.5:24.8f}")
    assert energy >= -0.5 - 1e-10
for name, zeta in [("STO zeta=1.24", 1.24), ("Exact H 1s", 1.0)]:
    energy = zeta**2 / 2 - zeta
    print(f"{name:18s} {energy:18.8f} {energy + 0.5:24.8f}")
# Independent reference values detect missing primitive normalization or Coulomb factors.
np.testing.assert_allclose(energies["STO-3G"], -0.46658185, atol=2e-7, rtol=0)
np.testing.assert_allclose(energies["STO-6G"], -0.47103905, atol=2e-7, rtol=0)

The six-primitive contraction gives a lower one-electron trial energy for this example, approaching the energy of its Slater shape. That shape still has $\zeta=1.24$, rather than the isolated-H optimum $\zeta=1$.

The variational principle does not guarantee monotonically lower energies when switching between arbitrary named basis sets. A non-increasing variational minimum is guaranteed when the new trial space contains the old one and both optimizations are performed for the same Hamiltonian. Here the two contracted one-function spaces are not nested.

### 4.7.1. Research practice: test flexibility against a known answer

Suppose you are checking whether a compact atomic basis is limiting a calculation. Adding more Gaussian primitives inside **one fixed contraction** and giving their coefficients **independent freedom** are different changes. We can isolate this distinction using the known hydrogen energy, without experimental uncertainty or electron correlation.

Keep every published exponent unchanged. For each STO-nG record, compare its fixed contraction with the lowest-energy normalized linear combination of its individual primitives. The latter is a deliberately **uncontracted teaching space**, not the published STO-nG basis. Solve $Hc=ESc$: $H$ is the one-electron Hamiltonian matrix, $S$ is overlap, $c$ contains variable coefficients, and $E$ is a trial energy. [SciPy's generalized Hermitian eigensolver](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.eigh.html) performs this small optimization.

The fixed contraction lies inside its corresponding uncontracted space, so the optimized energy cannot increase. There is no such nesting guarantee between arbitrary named basis sets. In a larger molecular study, this sort of controlled comparison helps distinguish extra representation freedom from changes in the electronic method.

In [ ]:
basis_validation = []
for name, parameters in basis.items():
    hamiltonian, overlap = hydrogen_matrices(parameters)
    eigenvalues, eigenvectors = eigh(hamiltonian, overlap)
    best_energy = float(eigenvalues[0])
    best_coefficients = eigenvectors[:, 0]
    np.testing.assert_allclose(best_coefficients @ overlap @ best_coefficients, 1, atol=1e-10)
    assert -0.5-1e-10 <= best_energy <= energies[name]+1e-10
    basis_validation.append((name, energies[name], best_energy, len(parameters["alpha"])))

fig, ax = plt.subplots(figsize=(8, 3.7), layout="constrained")
positions = np.arange(len(basis_validation))
ax.bar(positions-0.18, [row[1]+0.5 for row in basis_validation], width=0.35, label="Published fixed contraction")
ax.bar(positions+0.18, [row[2]+0.5 for row in basis_validation], width=0.35, label="Optimize primitive coefficients")
ax.set_xticks(positions, [f"{row[0]} exponents\n1 fixed function vs {row[3]} independent functions" for row in basis_validation])
ax.set(ylabel="Energy above exact H ground state (hartree)", title="Which extra freedom lowers the hydrogen error?")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.2)
fig.savefig(OUTPUT_DIR / "basis_flexibility_check.png", dpi=130, bbox_inches="tight")
plt.show()
for name, fixed, free, count in basis_validation:
    print(f"{name}: fixed {fixed:.8f}; {count}-function optimum {free:.8f} hartree")

**What decision does this support?** The observed improvement comes from freeing radial coefficients in the same one-electron Hamiltonian. It does not measure molecular correlation, solvent effects, or a universal basis ranking. The next sensible test depends on the property: molecular bonding may also need polarization and diffuse functions, not merely more s-type radial freedom.

**Check your understanding:** why must each orange bar be no higher than its blue partner? Because its variational space contains the fixed contraction. Why can both bars remain above zero? Because a finite Gaussian space still restricts the exact hydrogen function.

## 4.8. Practice and self-check

1. For three electrons and two nuclei, count electron-electron, nucleus-nucleus, and electron-nucleus terms. Why is the electron-electron sum written with $i<j$?
2. Does Born-Oppenheimer separation eliminate vibrational zero-point energy? Which part of the model would describe it?
3. Explain why the many-electron electronic Hamiltonian cannot generally be used as a one-electron MO equation.
4. Remove the Gaussian primitive normalization factors in a copy of the calculation. Which check detects the changed amplitude, and why can unchanged analytic energy code still return its original result?
5. Why does dividing a curve by its maximum fail to normalize a three-dimensional orbital?
6. For an STO with $\zeta=2$ $bohr^{-1}$, predict the radial peak and mean radius. Verify them by plotting or quadrature.
7. Does hydrogen STO-6G supply six independent variational functions? Explain the difference between a primitive count and a contracted basis-function count.
8. Explain why STO-6G can fit a chosen Slater shape better while still giving an inaccurate isolated-H energy.
9. Optional: replace $\alpha_i$ by $\alpha_i/1.24^2$, renormalize the primitives and contraction, and repeat the H comparison. Explain how a spatial rescaling changes Gaussian exponents quadratically.

<details><summary>Selected answers</summary>

1. Three electron-electron, one nucleus-nucleus, and six electron-nucleus terms. The ordered restriction prevents self-interactions and double counting.
2. No. Nuclear quantum motion in $\hat T_N+U(\mathbf R)$ produces vibrational energy levels and zero-point energy.
3. Its electron-electron interactions act on a function of all electronic coordinates; Hartree-Fock and Kohn-Sham equations use effective one-electron operators instead.
4. The evaluated curve no longer matches the normalized primitives used by the overlap and energy formulas; the independent norm integral detects the mismatch. Unchanged analytic energy formulas still describe the original normalized primitives, so they can pass even when the amplitude evaluator is wrong.
5. Normalization constrains an integral of the squared amplitude over volume, not its height.
6. The most probable radius is $1/\zeta=0.5$ bohr and the mean radius is $3/(2\zeta)=0.75$ bohr.
7. No: six fixed primitives combine into one contracted s function for H.
8. Shape fitting and energy minimization are different tasks, and the supplied contraction approximates an exponent near 1.24 rather than the exact isolated-H exponent 1.
9. The rescaling makes these H contractions approximate a $\zeta\approx1$ Slater function, but the modified data are no longer the published, unmodified H STO-nG basis.

</details>

## References and next steps

The linked documentation supports the implementations and conventions; it is useful to record the exact basis version as well as its name.

- [Basis Set Exchange](https://www.basissetexchange.org/) and [BSE command-line documentation](https://molssi-bse.github.io/basis_set_exchange/bse_cli.html): basis records, versioning, and original references. The BSE source records attribute these STO-nG data to Hehre, Stewart, and Pople (1969), *Self-Consistent Molecular-Orbital Methods. I. Use of Gaussian Expansions of Slater-Type Atomic Orbitals*, DOI [10.1063/1.1672392](https://doi.org/10.1063/1.1672392).
- [GBasis documentation](https://gbasis.qcdevs.org/_autosummary/gbasis.html): primitive and contraction conventions.
- [Psi4 Hartree-Fock theory](https://psi4.github.io/psi4docs/master/scf.html): orbitals, density matrices, self-consistency, and energy expressions.
- [Born and Oppenheimer (1927)](https://doi.org/10.1002/andp.19273892002): original separation of electronic and nuclear motion.
- [NIST units and constants](https://www.nist.gov/publications/units-and-constants): atomic-unit conventions.

**Takeaway:** specify the Hamiltonian, approximations, basis, normalization, and quantity being compared. A better-looking orbital plot is not enough to establish a more accurate calculation. Continue with [Chapter 5](Chapter05.ipynb).